In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize
from scipy.special import logsumexp, softmax
from time import perf_counter


In [ ]:

RNG = np.random.default_rng(0)
NB, NI = 48, 40		# boundary and interior points per class
_T = np.linspace(0, 2*np.pi, NB, endpoint=False)
_U = RNG.uniform(size=(NI, 2))

def rot(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s], [s, c]])

def blobs(a, b, width=0.3, wobble=0.15):
    'Return the classes as a list of (88, 2) arrays.'
    cx, L = (a + b)/2, (b - a)/2
    r = 1 + wobble*(0.5*np.sin(3*_T) + 0.3*np.cos(5*_T + 1))
    bd = np.stack([cx + L*r*np.cos(_T), 1 + width*r*np.sin(_T)], 1)
    rr, th = np.sqrt(_U[:, 0])*0.9, 2*np.pi*_U[:, 1]
    rin = 1 + wobble*(0.5*np.sin(3*th) + 0.3*np.cos(5*th + 1))
    inn = np.stack([cx + L*rr*rin*np.cos(th), 1 + width*rr*rin*np.sin(th)], 1)
    base = np.vstack([bd, inn])
    return [base @ rot(2*np.pi*i/3).T for i in range(3)]

def lift(P):
    'Augment samples: x -> (x, 1).'
    return np.hstack([P, np.ones((len(P), 1))])

def dataset(C):
    'Augmented samples X and labels y of the classes C.'
    X = np.vstack([lift(P) for P in C])
    y = np.concatenate([[i]*len(P) for i, P in enumerate(C)])
    return X, y


In [ ]:

def sep_lp(C, strict):
    rows = []
    for i in range(3):
        Z = lift(C[i])
        for j in range(3):
            if j != i:
                G = np.zeros((len(Z), 9))
                G[:, 3*i:3*i+3] += Z
                G[:, 3*j:3*j+3] -= Z
                rows.append(G)
    G = np.vstack(rows)		# each row gives y_i - y_j
	# centering: w_1 + w_2 + w_3 = 0
    cen = [np.tile(np.eye(3)[k], 3) for k in range(3)]
    free = [(None, None)]*9
    if strict:
        res = linprog(np.zeros(9), A_ub=-G, b_ub=-np.ones(len(G)),
                      A_eq=np.vstack(cen), b_eq=np.zeros(3), bounds=free, method='highs')
    else:
        res = linprog(np.zeros(9), A_ub=-G, b_ub=np.zeros(len(G)),
                      A_eq=np.vstack([G.sum(0)] + cen), b_eq=[1, 0, 0, 0], bounds=free, method='highs')
    return res.x.reshape(3, 3) if res.status == 0 else None

def hulls_disjoint(A, B):
    'True if the convex hulls of A and B are strictly separated by a line.'
    Z = np.vstack([lift(A), -lift(B)])
    res = linprog(np.zeros(3), A_ub=-Z, b_ub=-np.ones(len(Z)), bounds=[(None, None)]*3, method='highs')
    return res.status == 0

def pair_line(A, B):
    'Maximum-margin line w = (m, b), with m.x + b >= 1 on A and <= -1 on B, or None if none exists.'
    Z = np.vstack([lift(A), -lift(B)])
    lp = linprog(np.zeros(3), A_ub=-Z, b_ub=-np.ones(len(Z)), bounds=[(None, None)]*3, method='highs')
    if lp.status != 0:
        return None
    res = minimize(lambda w: w[0]**2 + w[1]**2, lp.x, jac=lambda w: np.array([2*w[0], 2*w[1], 0.0]),
                   constraints=[{'type': 'ineq', 'fun': lambda w: Z @ w - 1, 'jac': lambda w: Z}],
                   method='SLSQP')
    return res.x


In [ ]:

def gaps(W, X, y):
    'For each sample, its own level minus the highest other level.'
    Y = X @ W.T
    rows = np.arange(len(y))
    own = Y[rows, y].copy()
    Y[rows, y] = -np.inf
    return own - Y.max(1)

def mismatched(W, X, y):
    'Boolean array: which samples have y_i <= y_j for some j != i.'
    return gaps(W, X, y) <= 0

def margin(W, X, y):
    'Smallest gap y_i - y_j of the unit weight W/|W|.'
    return gaps(W/np.linalg.norm(W), X, y).min()


In [ ]:

def gd(X, y, iters=200_000):
    Pm = np.eye(3)[y]	# one-hot targets
    N = len(X)
    t = 1/np.mean(np.sum(X**2, 1))
    W = np.zeros((3, 3))
    norms, losses = [], []
    sep = None		# (n, W_n, seconds) at first separation
    rec = set(np.unique(np.logspace(0, np.log10(iters), 300).astype(int)))
    t0 = perf_counter()
    for n in range(1, iters + 1):
        Q = softmax(X @ W.T, axis=1)
        W -= t*(Q - Pm).T @ X/N
        if sep is None and not mismatched(W, X, y).any():
            sep = (n, W.copy(), perf_counter() - t0)
        if n in rec:
            Y = X @ W.T
            norms.append((n, np.linalg.norm(W)))
            losses.append((n, np.mean(logsumexp(Y, axis=1) - np.sum(Pm*Y, 1))))
    return W, np.array(norms), np.array(losses), sep, perf_counter() - t0


In [ ]:

PARAMS = [(-2, 0.3), (-2.5, 1.1), (-2.5, 1.3)]
runs = {}
for a, b in PARAMS:
    C = blobs(a, b)
    X, y = dataset(C)
    separable = sep_lp(C, strict=False) is not None
    strictly = sep_lp(C, strict=True) is not None
    disjoint = all(hulls_disjoint(C[i], C[j]) for i, j in [(0, 1), (0, 2), (1, 2)])
    W, norms, losses, sep, seconds = gd(X, y)
    grad = (softmax(X @ W.T, axis=1) - np.eye(3)[y]).T @ X/len(X)
    runs[(a, b)] = dict(C=C, X=X, y=y, W=W, norms=norms, losses=losses, sep=sep, seconds=seconds, separable=separable)
    print(f'(a,b) = ({a}, {b}): separable = {separable}, strictly = {strictly}, '
          f'hulls pairwise disjoint = {disjoint}')
    print(f'   after {int(norms[-1, 0])} steps: |W| = {norms[-1, 1]:.2f}, loss = {losses[-1, 1]:.2e}, '
          f'|grad| = {np.linalg.norm(grad):.1e}, '
          f'misclassified = {int(mismatched(W, X, y).sum())} of {len(X)}, {seconds:.1f} s')
    print(f'   first separation: ' + (f'n = {sep[0]}' if sep else 'never'))


In [ ]:

plt.rcParams.update({'font.family': 'serif',
                     'font.serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
                     'mathtext.fontset': 'stix', 'font.size': 8, 'axes.linewidth': 0.6,
                     'pdf.fonttype': 42})
cols = ['#c0392b', '#2e8b3a', '#2f5fb3']
marks = ['o', '^', 's']
tints = ['#f6dcd9', '#dcefdc', '#dce5f5']
LIM = 2.9
g = np.linspace(-LIM, LIM, 500)
X1, X2 = np.meshgrid(g, g)
GRID = np.stack([X1.ravel(), X2.ravel(), np.ones(X1.size)], 1)
BOX = [(-LIM, -LIM), (LIM, -LIM), (LIM, LIM), (-LIM, LIM)]

def clip(poly, a, c):
    'Clip the convex polygon poly to the half-plane a.x + c >= 0.'
    out = []
    for k in range(len(poly)):
        p, q = np.array(poly[k]), np.array(poly[(k + 1) % len(poly)])
        fp, fq = a @ p + c, a @ q + c
        if fp >= 0:
            out.append(p)
        if fp*fq < 0:		# the edge crosses the line
            out.append(p + fp/(fp - fq)*(q - p))
    return out

def region(W, i):
    'The region R_i of W inside the box, as a polygon; on ties the lower index wins.'
    poly = BOX
    for j in range(len(W)):
        if j != i:
            a, c = W[i, :2] - W[j, :2], W[i, 2] - W[j, 2]
            if np.allclose(a, 0) and np.isclose(c, 0) and j < i:
                return []		# identical rows
            poly = clip(poly, a, c)
            if not poly:
                return []
    return poly

def regions(ax, C, W, title):
    'Shade the regions of W and plot the three classes.'
    for i in range(len(W)):
        poly = region(W, i)
        if poly:
            ax.add_patch(plt.Polygon(poly, closed=True, facecolor=tints[i],
                                     edgecolor=tints[i], linewidth=0.3, zorder=0))
    for P, c, m in zip(C, cols, marks):
        ax.scatter(P[:, 0], P[:, 1], s=4, c=c, marker=m, linewidths=0)
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, 2.3); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=8)

fig, axs = plt.subplots(1, 3, figsize=(6.9, 2.5))
for ax, (a, b), lab in zip(axs, PARAMS, 'abc'):
    r = runs[(a, b)]
    if r['sep'] is not None:	# stop at first separation
        n_sep, W, _ = r['sep']
        sub = f'separable, stopped at $n={n_sep}$'
    else:
        W, sub = r['W'], 'not separable, $n=2\\times10^5$'
    regions(ax, r['C'], W, f'({lab}) $a={a},\\ b={b}$\n{sub}')
    if not r['separable']:		# pairwise separating lines
        for i, j in [(0, 1), (1, 2), (2, 0)]:
            w = pair_line(r['C'][i], r['C'][j])
            if w is not None:
                ax.contour(X1, X2, (GRID @ w).reshape(X1.shape), levels=[0],
                           colors='k', linestyles='--', linewidths=0.7)
plt.tight_layout(pad=0.3, w_pad=0.8)
plt.savefig('pinwheel_gd.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()
